In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import mph
import mphsweepkit as msk

In [3]:
# Start the COMSOL client
client = mph.start()

In [4]:
# Load the model
model = client.load('core_cross_section_solved.mph')

Load an already solved CascadedSweepModel

In [5]:
csm = msk.CascadedSweepModel(model, 'Study on Cross-Sections', show_param_names=True)

Initialized CascadedSweepModel
Study name: Study on Cross-Sections
Sweep Structure:
    - Geometry Sweep (BatchSweep) -> 'hor_slit', 'vert_slit', 'w', 'l_r', 'a_e'
      - Material Sweep (MaterialSweep) -> 'matsw.comp1.core'
        - Excitation Sweep (Parametric) -> 'b_mean'
          - Frequency Sweep (Frequency)
Loop names: ['Geometry Sweep', 'Material Sweep', 'Excitation Sweep', 'Frequency Sweep']
Loop lengths: [3, 1, 4, 13]
--------------------------------
Data updated from MPh-model.
Input data shape: (156, 8)
Reset output data to shape of the input data: (156, 0)
Combined shape: (156, 8)


Post-processing of global equations

In [7]:
# Load the customized post-processing expressions from a JSON file
post_processing_exprs = msk.load_post_processing_exprs("global_post_processing_expressions.json", print_info=False)

# Perform post-processing of global data
csm.post_process_globals(post_processing_exprs)

# Show the first few rows of the combined data
csm.combined_data.head()

name,hor_slit,vert_slit,w,l_r,a_e,matsw.comp1.core,b_mean,freq,p_loss,p_mag,p_el
unit,um,um,mm,mm,mm,,mT,kHz,W /m^3,W /m^3,W /m^3
group,Geometry Sweep,Geometry Sweep,Geometry Sweep,Geometry Sweep,Geometry Sweep,Material Sweep,Excitation Sweep,Frequency Sweep,post-processing,post-processing,post-processing
0,0.0,0.0,10.0,0.0,5.0,2.0,25.0,100.0,1138.793175,1016.234899,122.558276
1,0.0,0.0,10.0,0.0,5.0,2.0,25.0,200.0,3233.052526,2542.864479,690.188047
2,0.0,0.0,10.0,0.0,5.0,2.0,25.0,300.0,6623.256480,4630.243339,1993.013141
3,0.0,0.0,10.0,0.0,5.0,2.0,25.0,400.0,11906.198592,7588.854681,4317.343911
4,0.0,0.0,10.0,0.0,5.0,2.0,25.0,500.0,19822.375039,11867.487161,7954.887878


In [8]:
# Save the input and output dataframes to CSV files
csm.save_global_data()

Saved result data to: X:\Till_data\repositories\MPhSweepKit\examples\studies\infinite_ferrite_cross_sections\global_data


Post-processing of fields on a selected domain

In [12]:
# All available selections can be printed with
# csm.print_available_selections()

# Select a domain selection for post-processing.
# The selection name must exist in the COMSOL model:
selection_name = "Core"
selection_type = "dom"
selection_domain_tag = csm._get_selection_tag(selection_name, selection_type="dom")
print(f"Selected domain tag: {selection_domain_tag}")

# Choose a name for the dataset that will be created from the selection:
selection_dataset_name = "Solution Core"

# Create a dataset from the selection
csm.create_dataset_selection(selection_name, selection_type, selection_dataset_name)

# A dataset can be removed with
# csm.node_datasets.children()[-1].remove()

Selected domain tag: geom1_csel1_dom
Creating dataset 'Solution Core' from selection 'Core' of type 'dom'.


In [15]:
# Evaluate field expressions on the selection dataset and export the results to text files.
#   For each geometry configuration, a separate text file is created with the corresponding results.
#   This allows mesh reuse for expression evaluation and straightforward result comparison.

# The loop levels for the COMSOL export can be obtained with
comsol_looplevels = csm.get_comsol_looplevels()

# Load customized post-processing expressions
post_processing_exprs = msk.load_post_processing_exprs("fields_post_processing_expressions.json", print_info=False)

# Convert the post-processing expressions into lists for the COMSOL export
expressions = [entry["expression"] for entry in post_processing_exprs.values()]
labels = [entry["label"] for entry in post_processing_exprs.values()]
units = [entry["unit"] for entry in post_processing_exprs.values()]

# TODO: Add a visualization of the different geometries

# Iterate over the loop levels and export the results for each geometry configuration
for looplevel_arr in comsol_looplevels:

    export_node = csm.export_dataset_with_expressions(
        dataset_name=selection_dataset_name,
        expressions=expressions,
        descriptions=labels,
        units=units,
        looplevel=looplevel_arr,
        looplevelinput=['manual'] * len(looplevel_arr)
    )

    print(f"Loop level: {export_node.property('looplevel')}\n")

Dataset to be exported:             'Solution Core'
Overwrite existing export node:     'Expressions on Solution Core'
List of expressions:                ['mf.normB', 'mf.normE']
Loop level: [array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.])
 array([1., 2., 3., 4.]) array([1.])]

Dataset to be exported:             'Solution Core'
Overwrite existing export node:     'Expressions on Solution Core'
List of expressions:                ['mf.normB', 'mf.normE']
Loop level: [array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.])
 array([1., 2., 3., 4.]) array([2.])]

Dataset to be exported:             'Solution Core'
Overwrite existing export node:     'Expressions on Solution Core'
List of expressions:                ['mf.normB', 'mf.normE']
Loop level: [array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.])
 array([1., 2., 3., 4.]) array([3.])]



Release Model

In [16]:
client.remove(model)
client.clear()
client.disconnect()